# Heatmap Generation — SQ-MIL (heatmap-only)

Overlays **already-computed** per-patch predictions (`inst_predictions_fold*.csv`) onto
the real `.tif` WSI tissue thumbnails. No model / checkpoint / GPU required.

**Prerequisites (all on your local machine):**
1. `inst_predictions_fold0.csv` … `inst_predictions_fold4.csv` — download from Drive
   `results/stage2/foldN/` into the matching local folders (see PREDICTIONS_DIR below).
   Across the 5 folds every slide appears exactly once, so all five together cover the whole dataset.
2. Your WSI `.tif` files, each named `{slide_id}.tif` (e.g. `1289.tif`).

Color scheme: CC (Red), EC (Green), HGSC (Yellow), LGSC (Blue), MC (Purple)

In [ ]:
# ============================================================
# CONFIGURATION — edit these paths before running
# ============================================================

# Project root. Leave as None to use the current working directory
# (run this notebook from C:\Users\zpanp\projects\SQ-MIL).
PROJECT_ROOT_OVERRIDE = None

# Directory containing the instance-prediction CSVs. Point at the parent
# 'results/stage2' folder (which holds fold0..fold4 subdirs, each with an
# inst_predictions_foldN.csv) to render EVERY slide in one pass.
# You can also point at a single fold dir or a single .csv file.
PREDICTIONS_DIR = 'results/stage2'

# Directory (or glob) with your local WSI .tif files, named {slide_id}.tif
WSI_DIR         = 'data/wsi'

# Where to write the heatmap PNGs
OUTPUT_DIR      = 'results/heatmaps'

# --- Heatmap appearance ---
THUMBNAIL_SIZE  = 2048
OVERLAY_ALPHA   = 0.50
GAUSSIAN_SIGMA  = 5.0
NUM_WORKERS     = 8

# --- Display ---
N_DISPLAY       = 6     # number of heatmaps to show inline

In [ ]:
# ============================================================
# SETUP + SANITY CHECK
# ============================================================
import sys, glob
from pathlib import Path

# Resolve project root (must contain src/)
if PROJECT_ROOT_OVERRIDE:
    PROJECT_ROOT = Path(PROJECT_ROOT_OVERRIDE).resolve()
else:
    PROJECT_ROOT = Path('.').resolve()
    if not (PROJECT_ROOT / 'src').is_dir() and (PROJECT_ROOT.parent / 'src').is_dir():
        PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src').is_dir():
    raise RuntimeError(f'src/ not found under {PROJECT_ROOT}; set PROJECT_ROOT_OVERRIDE.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.visualization.heatmap import HeatmapGenerator

def _abs(p):
    p = Path(p)
    return p if p.is_absolute() else (PROJECT_ROOT / p)

pred_path = _abs(PREDICTIONS_DIR)
wsi_path  = _abs(WSI_DIR)
out_path  = _abs(OUTPUT_DIR)
out_path.mkdir(parents=True, exist_ok=True)

# Count prediction CSVs
if pred_path.is_file():
    csvs = [pred_path]
else:
    csvs = sorted(pred_path.rglob('inst_predictions_fold*.csv')) or sorted(pred_path.rglob('*.csv'))

# Count WSIs
if '*' in str(wsi_path):
    wsis = sorted(Path(p) for p in glob.glob(str(wsi_path)))
elif wsi_path.is_dir():
    wsis = sorted(wsi_path.glob('*.tif')) + sorted(wsi_path.glob('*.tiff'))
else:
    wsis = [wsi_path] if wsi_path.exists() else []

print(f'Project root    : {PROJECT_ROOT}')
print(f'Prediction CSVs : {len(csvs)} found under {pred_path}')
for c in csvs[:6]:
    print(f'    - {c.relative_to(PROJECT_ROOT) if PROJECT_ROOT in c.parents else c}')
print(f'WSI .tif files  : {len(wsis)} found under {wsi_path}')
print(f'Output dir      : {out_path}')
if not csvs:
    print('\n[!] No prediction CSVs found — download inst_predictions_fold*.csv from Drive first.')
if not wsis:
    print('\n[!] No WSI .tif files found — heatmaps will fall back to a synthetic gray canvas.')

In [ ]:
# ============================================================
# (OPTIONAL) VERIFY slide_id <-> filename MATCHING
# Confirms the CSV slide_ids line up with your .tif filenames
# before spending time rendering.
# ============================================================
import pandas as pd

csv_ids = set()
for c in csvs:
    df_head = pd.read_csv(c, usecols=lambda col: col in ('slide_id', 'filename'))
    id_col = 'slide_id' if 'slide_id' in df_head.columns else 'filename'
    csv_ids |= set(df_head[id_col].astype(str).str.replace(r'\.tif+$', '', regex=True).unique())

wsi_ids = {p.stem for p in wsis}
matched = csv_ids & wsi_ids
print(f'Unique slides in CSVs : {len(csv_ids)}')
print(f'Unique WSIs on disk   : {len(wsi_ids)}')
print(f'Matched (will render with tissue background) : {len(matched)}')
missing_wsi = sorted(csv_ids - wsi_ids)[:10]
if missing_wsi:
    print(f'CSV slides with no matching .tif (synthetic canvas): {missing_wsi} ...')

In [ ]:
# ============================================================
# GENERATE HEATMAPS  (calls src.visualization.heatmap.HeatmapGenerator)
# ============================================================
gen = HeatmapGenerator(
    thumbnail_size = (THUMBNAIL_SIZE, THUMBNAIL_SIZE),
    overlay_alpha  = OVERLAY_ALPHA,
    gaussian_sigma = GAUSSIAN_SIGMA,
)

results = gen.generate_batch(
    wsi_dir         = str(wsi_path) if wsis else None,
    predictions_dir = str(pred_path),
    output_dir      = str(out_path),
    num_workers     = NUM_WORKERS,
)

ok   = sum(1 for v in results.values() if not isinstance(v, Exception))
errs = {k: v for k, v in results.items() if isinstance(v, Exception)}
print(f'\nGenerated: {ok} heatmaps -> {out_path}')
if errs:
    print(f'Failed: {len(errs)}')
    for sid, exc in list(errs.items())[:5]:
        print(f'  {sid}: {exc}')

In [ ]:
# ============================================================
# DISPLAY HEATMAPS INLINE
# ============================================================
from IPython.display import Image, display

heatmap_files = sorted(glob.glob(str(out_path / '*_heatmap.png')))
print(f'Total heatmap files: {len(heatmap_files)}')
for hf in heatmap_files[:N_DISPLAY]:
    print(Path(hf).name)
    display(Image(hf, width=900))

In [ ]:
# ============================================================
# (OPTIONAL) SINGLE-SLIDE DEEP DIVE with custom settings
# ============================================================
# SLIDE_ID = '1289'
#
# df = pd.concat([pd.read_csv(c) for c in csvs], ignore_index=True)
# id_col = 'slide_id' if 'slide_id' in df.columns else 'filename'
# slide_df = df[df[id_col].astype(str).str.replace(r'\.tif+$', '', regex=True) == SLIDE_ID].reset_index(drop=True)
# print(f'Patches for {SLIDE_ID}: {len(slide_df)}')
#
# gen_custom = HeatmapGenerator(thumbnail_size=(4096, 4096), overlay_alpha=0.60, gaussian_sigma=3.0)
# out_png = out_path / f'{SLIDE_ID}_heatmap_custom.png'
# gen_custom.generate(
#     wsi_path       = wsi_path / f'{SLIDE_ID}.tif',
#     predictions_df = slide_df,
#     output_path    = out_png,
# )
# display(Image(str(out_png), width=1200))